# AGender - Step 3: which attributes can be released? (CPU)

The baseline QUBO selector, with relevance aimed at the partner's task (data demand, `INTERNET_MB`) and a
privacy term added: what each attribute, and each pair of attributes, reveals about gender and age group.

- **Defences compared:** release everything; remove the most revealing attributes one at a time; the QUBO without
  the privacy term (gamma = 0, the baseline selector); the privacy-aware QUBO (k x gamma grid); coarsening
  every attribute to a few levels; adding noise.
- **Every candidate release** is scored by utility (LightGBM R^2 for log data demand) and leakage (test AUC of
  the attacker the partner picks on validation, retrained on what is released; 1 = as much as releasing everything,
  0 = guessing).
- **Operating point** of each defence, chosen on validation: the lowest leakage that keeps at least 90 % of the
  utility. Each operating point then gets the full Step 2B audit (five attackers, 90 % conformal sets).
- **Solver check:** on the first repeat, the exact minimum of the energy over all 2^24 subsets, compared with the
  baseline single-flip annealer and with the annealer with swap moves used here.

**Kaggle settings:** Accelerator = **None** (CPU). Internet = Off is fine.
**Inputs to add:** `agender-code` and `agender-step1`.
**Expected run time:** about 50-70 minutes.


In [ ]:
import os
import sys
from pathlib import Path

# The AGender code is attached as a Kaggle dataset. Find the `agender` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('audit.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender').exists():
        os.symlink(package.parent, code_root / 'agender')

sys.path.insert(0, str(code_root))
import agender
print('AGender code loaded from', Path(agender.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/agender_outputs/step3')
TIME_BUDGET_MINUTES = 240
REPEATS = range(5)
EXACT_CHECK = True


In [ ]:
import json

import pandas as pd

from agender import environment, step3
from agender.runner import StepRunner

runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}
# Tables and splits come from the saved output of Step 1 (agender-step1).
files = {name: find_file(name, INPUT) for name in
         ('lat_clean.csv', 'lat_splits.csv', 'copenhagen_people.csv', 'copenhagen_splits.csv')}
print({k: str(v) for k, v in files.items()})
if None in files.values():
    raise RuntimeError('Attach the saved output of Step 1 (agender-step1): it holds lat_clean.csv, '
                       'lat_splits.csv, copenhagen_people.csv and copenhagen_splits.csv.')

summary['environment'] = runner.run('measure the machine', environment.collect)
clean = pd.read_csv(files['lat_clean.csv'])
lat_splits = pd.read_csv(files['lat_splits.csv'])


In [ ]:
summary['release'] = runner.run('privacy-utility curves and operating points', step3.run, clean, lat_splits, OUTPUT,
                               repeats=REPEATS, exact=EXACT_CHECK)
if summary['release']:
    print(json.dumps(summary['release'], indent=1))


In [ ]:
summary['steps'] = runner.steps
with open(OUTPUT / 'step3_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)
print('Failed or skipped steps:', runner.failed() or 'none')
